# WQI Classification — Final Visualization & Diagnostics

This notebook is **classification-only** for the groundwater (GW) and surface-water (SW) experiments.

It uses:
- held-out **testing metrics** for model comparison;
- individual held-out **test predictions + probabilities** for confusion matrices and ROC–AUC;
- original physicochemical data for exploratory **PCA**;
- exploratory **chi-square (χ²) feature association scores** using discretized continuous predictors.

### Important methodological distinction
- **Testing metrics** are the primary generalisation results.
- **Training metrics** are diagnostic only.
- **PCA and χ² are exploratory analyses**, not additional classifiers.
- χ² is calculated after quantile-discretising continuous water-quality variables. It is **not** used to fit the reported classifiers in this notebook.


In [ ]:
from pathlib import Path
import warnings
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc
)
from scipy.stats import chi2_contingency

warnings.filterwarnings("ignore")

BASE = Path.cwd()
PLOTS = BASE / "classification_plots_final"
PLOTS.mkdir(exist_ok=True)

print("Working directory:", BASE)
print("Output folder:", PLOTS)


In [ ]:
# Locate files robustly
def locate(*names):
    roots = [BASE, BASE.parent, BASE / "data", BASE / "results"]
    for root in roots:
        for name in names:
            p = root / name
            if p.exists():
                return p
    return None

def read_optional(*names):
    p = locate(*names)
    if p is None:
        print("Not found:", names)
        return None
    print("Loaded:", p)
    return pd.read_csv(p)

gw_train = read_optional("gw_training_result.csv")
gw_test  = read_optional("gw_testing_result.csv")
sw_train = read_optional("sw_training_result.csv")
sw_test  = read_optional("sw_testing_result.csv")

gw_data = read_optional("gw_sanity.csv")
sw_data = read_optional("surface_wqi_pred.csv")

print("\nSearching for individual test-prediction files...")
gw_pred_files = sorted(BASE.glob("gw_*_test_predictions.csv"))
sw_pred_files = sorted(BASE.glob("sw_*_test_predictions.csv"))

print("GW prediction files:", [p.name for p in gw_pred_files])
print("SW prediction files:", [p.name for p in sw_pred_files])


In [ ]:
# Standardise result-column names so the plotting code accepts the saved CSVs.
def standardize_result_columns(df):
    if df is None:
        return None
    rename = {}
    for c in df.columns:
        k = str(c).strip().lower().replace("_", " ")
        if k in {"model", "algorithm", "classifier"}:
            rename[c] = "Model"
        elif "accuracy" in k:
            rename[c] = "Accuracy"
        elif "precision" in k:
            rename[c] = "Precision"
        elif "recall" in k:
            rename[c] = "Recall"
        elif "f1" in k:
            rename[c] = "F1"
        elif k == "mcc" or "matthews" in k:
            rename[c] = "MCC"
        elif "cv" in k and "acc" in k:
            rename[c] = "5-Fold CV Acc"
    return df.rename(columns=rename)

gw_train = standardize_result_columns(gw_train)
gw_test = standardize_result_columns(gw_test)
sw_train = standardize_result_columns(sw_train)
sw_test = standardize_result_columns(sw_test)

def best_row(df):
    if df is None or "Model" not in df.columns:
        return None
    metric = "F1" if "F1" in df.columns else "Accuracy"
    if metric not in df.columns:
        return None
    s = pd.to_numeric(df[metric], errors="coerce")
    return df.loc[s.idxmax()]

for label, df in [("GW testing", gw_test), ("SW testing", sw_test)]:
    r = best_row(df)
    if r is not None:
        print(f"{label} best model by testing F1: {r['Model']}")


## 1. Original WQI class distribution

In [ ]:
def detect_class_column(df):
    if df is None:
        return None
    candidates = [
        "wqi_class", "WQI_Class", "WQI Class", "WQI_Category",
        "wqi category", "Class", "class", "label", "Label"
    ]
    for c in candidates:
        if c in df.columns:
            return c
    return None

def ensure_wqi_class(df):
    if df is None:
        return None
    df = df.copy()
    if detect_class_column(df) is not None:
        return df

    # If a WQI column exists but class labels do not, reproduce the class thresholds
    # used in the classification notebooks.
    wqi_col = next((c for c in df.columns if str(c).lower() == "wqi"), None)
    if wqi_col is not None:
        def wqi_class(wqi):
            if pd.isna(wqi): return np.nan
            if wqi <= 50: return "Excellent"
            if wqi <= 100: return "Good"
            if wqi <= 200: return "Poor"
            if wqi <= 300: return "Very Poor"
            return "Unsuitable"
        df["wqi_class"] = df[wqi_col].apply(wqi_class)
    return df

gw_data = ensure_wqi_class(gw_data)
sw_data = ensure_wqi_class(sw_data)

def class_count_plot(df, title, filename):
    if df is None:
        print("Skipped:", title)
        return
    c = detect_class_column(df)
    if c is None:
        print("Skipped:", title, "— WQI class column not detected.")
        return

    counts = df[c].astype(str).value_counts()
    fig, ax = plt.subplots(figsize=(9,5))
    bars = ax.bar(counts.index, counts.values)
    ax.set_xlabel("WQI Class")
    ax.set_ylabel("Number of observations")
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=25)
    for b, v in zip(bars, counts.values):
        ax.text(b.get_x()+b.get_width()/2, v, str(v), ha="center", va="bottom")
    plt.tight_layout()
    plt.savefig(PLOTS / filename, dpi=300, bbox_inches="tight")
    plt.show()

class_count_plot(gw_data, "Groundwater — Original WQI Class Distribution", "GW_class_distribution.png")
class_count_plot(sw_data, "Surface Water — Original WQI Class Distribution", "SW_class_distribution.png")


## 2. Testing performance — primary model comparison

In [ ]:
def classification_bar(gw, sw, metric, filename):
    if gw is None or sw is None or metric not in gw.columns or metric not in sw.columns:
        print("Skipped:", metric)
        return

    a = gw[["Model", metric]].rename(columns={metric:"Groundwater"})
    b = sw[["Model", metric]].rename(columns={metric:"Surface Water"})
    m = pd.merge(a, b, on="Model", how="outer")

    x = np.arange(len(m))
    width = 0.38
    fig, ax = plt.subplots(figsize=(11,6))
    ax.bar(x-width/2, m["Groundwater"], width, label="Groundwater")
    ax.bar(x+width/2, m["Surface Water"], width, label="Surface Water")
    ax.set_xticks(x)
    ax.set_xticklabels(m["Model"], rotation=30, ha="right")
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1.05)
    ax.set_title(f"Testing {metric} — GW vs SW")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.25)
    plt.tight_layout()
    plt.savefig(PLOTS / filename, dpi=300, bbox_inches="tight")
    plt.show()

for metric in ["Accuracy", "Precision", "Recall", "F1", "MCC"]:
    classification_bar(gw_test, sw_test, metric, f"testing_{metric}.png")


## 3. Training vs testing accuracy — overfitting diagnostic

In [ ]:
def train_test_accuracy_plot(train_df, test_df, region, filename):
    if train_df is None or test_df is None:
        print("Skipped:", region)
        return
    if "Accuracy" not in train_df.columns or "Accuracy" not in test_df.columns:
        return

    a = train_df[["Model","Accuracy"]].rename(columns={"Accuracy":"Training"})
    b = test_df[["Model","Accuracy"]].rename(columns={"Accuracy":"Testing"})
    m = pd.merge(a, b, on="Model", how="outer")

    x = np.arange(len(m))
    width = 0.38
    fig, ax = plt.subplots(figsize=(11,6))
    ax.bar(x-width/2, m["Training"], width, label="Training")
    ax.bar(x+width/2, m["Testing"], width, label="Testing")
    ax.set_xticks(x)
    ax.set_xticklabels(m["Model"], rotation=30, ha="right")
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0, 1.05)
    ax.set_title(f"{region} — Training vs Testing Accuracy")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.25)
    plt.tight_layout()
    plt.savefig(PLOTS / filename, dpi=300, bbox_inches="tight")
    plt.show()

train_test_accuracy_plot(gw_train, gw_test, "Groundwater", "GW_training_vs_testing_accuracy.png")
train_test_accuracy_plot(sw_train, sw_test, "Surface Water", "SW_training_vs_testing_accuracy.png")


## 4. Radar plot — best model in each water domain

In [ ]:
def classification_radar(gw, sw, filename):
    metrics = ["Accuracy","Precision","Recall","F1","MCC"]
    if gw is None or sw is None:
        return
    if not all(m in gw.columns and m in sw.columns for m in metrics):
        return

    g, s = best_row(gw), best_row(sw)
    angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]

    gv = [float(g[m]) for m in metrics] + [float(g[metrics[0]])]
    sv = [float(s[m]) for m in metrics] + [float(s[metrics[0]])]

    fig = plt.figure(figsize=(8,8))
    ax = fig.add_subplot(111, polar=True)
    ax.plot(angles, gv, marker="o", label=f"GW — {g['Model']}")
    ax.fill(angles, gv, alpha=0.10)
    ax.plot(angles, sv, marker="o", label=f"SW — {s['Model']}")
    ax.fill(angles, sv, alpha=0.10)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics)
    ax.set_ylim(0,1)
    ax.set_title("Best Testing Classifier — Multimetric Profile")
    ax.legend(loc="lower right", bbox_to_anchor=(1.25, 0.05))
    plt.tight_layout()
    plt.savefig(PLOTS / filename, dpi=300, bbox_inches="tight")
    plt.show()

classification_radar(gw_test, sw_test, "best_models_radar.png")


## 5. Confusion matrices — every exported classifier

These use **individual held-out test predictions** (`y_true`, `y_pred`). They cannot be reconstructed from Accuracy/F1/MCC summary tables.

## 4A. Model complexity radar — conceptual comparison

This radar is **not a measured performance result**. It summarizes relative algorithm characteristics on a 1–5 scale to make the methodological comparison easier to understand.

For a research paper, label it as a **conceptual/qualitative model-complexity profile**, not as an experimentally measured complexity score.


In [ ]:
# Relative algorithm-complexity profile (1 = low, 5 = high)
models = ["Random Forest", "Gradient Boosting", "XGBoost", "SVM", "KNN", "Decision Tree"]

# Qualitative scores: flexibility, tuning burden, computational cost,
# interpretability difficulty, and sensitivity to data scaling.
scores = {
    "Random Forest":     [4, 3, 3, 3, 2],
    "Gradient Boosting": [4, 4, 4, 4, 2],
    "XGBoost":           [5, 5, 4, 4, 2],
    "SVM":               [4, 4, 3, 4, 4],
    "KNN":               [3, 2, 2, 2, 5],
    "Decision Tree":     [3, 2, 2, 1, 1],
}

dims = ["Flexibility", "Tuning burden", "Computational cost",
        "Interpretability difficulty", "Scaling sensitivity"]

angles = np.linspace(0, 2*np.pi, len(dims), endpoint=False).tolist()
angles += angles[:1]

fig = plt.figure(figsize=(9, 9))
ax = fig.add_subplot(111, polar=True)

for model in models:
    vals = scores[model] + scores[model][:1]
    ax.plot(angles, vals, marker="o", label=model)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(dims, fontsize=9)
ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_title("Conceptual Model Complexity Profile", pad=20,
             fontsize=15, fontweight="bold")
ax.legend(loc="upper left", bbox_to_anchor=(1.05, 1.05), fontsize=8)

plt.tight_layout()
plt.savefig(PLOTS / "model_complexity_radar.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
def parse_model_from_filename(path):
    name = path.stem
    name = re.sub(r"^(gw|sw)_", "", name)
    name = re.sub(r"_test_predictions$", "", name)
    return name.replace("_", " ")

def plot_confusion_file(path, region):
    pred = pd.read_csv(path)
    if not {"y_true","y_pred"}.issubset(pred.columns):
        print("Skipped:", path.name, "— y_true/y_pred missing.")
        return

    y_true, y_pred = pred["y_true"], pred["y_pred"]
    labels = sorted(pd.unique(pd.concat([y_true, y_pred]).dropna()).tolist())
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    fig, ax = plt.subplots(figsize=(7,6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    model = parse_model_from_filename(path)
    ax.set_title(f"{region} — Confusion Matrix — {model}")
    plt.tight_layout()
    safe = re.sub(r"[^A-Za-z0-9]+", "_", model).strip("_")
    plt.savefig(PLOTS / f"{region}_CM_{safe}.png", dpi=300, bbox_inches="tight")
    plt.show()

for p in gw_pred_files:
    plot_confusion_file(p, "Groundwater")
for p in sw_pred_files:
    plot_confusion_file(p, "Surface Water")


## 5A. Combined confusion matrices — all classifiers in one high-resolution PNG

This creates one compact figure containing every available GW/SW test confusion matrix. It is designed for paper use: large canvas and 300 dpi.


In [ ]:
def combined_confusion_matrices(gw_files, sw_files, filename):
    files = [("Groundwater", p) for p in gw_files] + [("Surface Water", p) for p in sw_files]
    if not files:
        print("No test-prediction CSV files found.")
        return

    n = len(files)
    cols = 3
    rows = int(np.ceil(n / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(15, 4.8 * rows))
    axes = np.atleast_1d(axes).ravel()

    for ax, (region, path) in zip(axes, files):
        pred = pd.read_csv(path)

        if not {"y_true", "y_pred"}.issubset(pred.columns):
            ax.axis("off")
            ax.set_title(f"{region}\n{parse_model_from_filename(path)}\nmissing y_true/y_pred")
            continue

        labels = sorted(pd.unique(
            pd.concat([pred["y_true"], pred["y_pred"]]).dropna()
        ).tolist())

        cm = confusion_matrix(pred["y_true"], pred["y_pred"], labels=labels)

        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=labels
        )
        disp.plot(ax=ax, colorbar=False, cmap="Blues", values_format="d")

        ax.set_title(
            f"{region} — {parse_model_from_filename(path)}",
            fontsize=11,
            fontweight="bold"
        )
        ax.tick_params(axis="x", labelrotation=35, labelsize=8)
        ax.tick_params(axis="y", labelsize=8)

    for ax in axes[n:]:
        ax.axis("off")

    fig.suptitle(
        "Test-set Confusion Matrices — Groundwater and Surface Water",
        fontsize=16,
        fontweight="bold"
    )
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(PLOTS / filename, dpi=300, bbox_inches="tight")
    plt.show()

combined_confusion_matrices(
    gw_pred_files,
    sw_pred_files,
    "ALL_confusion_matrices.png"
)


## 6. Multiclass ROC–AUC — every exported classifier

ROC–AUC requires test-set probabilities or decision scores. The export cells in the final GW/SW classification notebooks save these values when the model supports them.

In [ ]:
def plot_roc_file(path, region):
    pred = pd.read_csv(path)
    if "y_true" not in pred.columns:
        return

    proba_cols = [c for c in pred.columns if str(c).startswith("y_proba_")]
    score_cols = [c for c in pred.columns if str(c).startswith("y_score_")]

    cols = proba_cols if len(proba_cols) >= 2 else score_cols
    if len(cols) < 2:
        print("Skipped ROC:", path.name, "— no usable probability/score columns.")
        return

    y_true = pred["y_true"]
    class_labels = [str(c).split("_", 2)[-1] for c in cols]

    # Convert encoded class labels to a stable common type.
    y_true_str = y_true.astype(str)
    available = [(c, lab) for c, lab in zip(cols, class_labels) if lab in set(y_true_str)]
    if len(available) < 2:
        print("Skipped ROC:", path.name, "— probability class labels do not match y_true.")
        return

    cols = [x[0] for x in available]
    class_labels = [x[1] for x in available]

    fig, ax = plt.subplots(figsize=(8,6))
    for col, cls in zip(cols, class_labels):
        y_bin = (y_true_str == cls).astype(int)
        scores = pd.to_numeric(pred[col], errors="coerce")
        valid = scores.notna()
        if valid.sum() == 0 or y_bin[valid].nunique() < 2:
            continue
        fpr, tpr, _ = roc_curve(y_bin[valid], scores[valid])
        score = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f"Class {cls} (AUC={score:.3f})")

    ax.plot([0,1], [0,1], linestyle="--", label="Chance")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{region} — Multiclass ROC–AUC — {parse_model_from_filename(path)}")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)
    plt.tight_layout()
    safe = re.sub(r"[^A-Za-z0-9]+", "_", parse_model_from_filename(path)).strip("_")
    plt.savefig(PLOTS / f"{region}_ROC_AUC_{safe}.png", dpi=300, bbox_inches="tight")
    plt.show()

for p in gw_pred_files:
    plot_roc_file(p, "Groundwater")
for p in sw_pred_files:
    plot_roc_file(p, "Surface Water")


## 7. PCA — exploratory WQI class separation

PCA is **not a classifier**. It projects the standardized physicochemical predictors into orthogonal directions of maximum variance.

For two components:

\[
Z = X_{standardized}W
\]

The explained-variance ratio is:

\[
EVR_k = \frac{\lambda_k}{\sum_j \lambda_j}
\]

The PCA plot helps answer: **do WQI classes occupy visibly different regions of feature space?**

It does **not** prove that a classifier will generalise well.


In [ ]:
def get_predictors(df):
    if df is None:
        return []
    exclude = {"wqi", "wqi_encoded", "wqi_class", "class", "label"}
    cols = []
    for c in df.select_dtypes(include=np.number).columns:
        if str(c).lower() in exclude:
            continue
        # Iron is directly used in the WQI formula in the classification workflow,
        # so exclude it from exploratory feature-space plots to avoid a direct WQI-derived signal.
        if "iron" in str(c).lower():
            continue
        cols.append(c)
    return cols

def pca_plot(df, title, filename, top_features=None):
    if df is None:
        return None

    class_col = detect_class_column(df)
    if class_col is None:
        print("Skipped:", title, "— class column missing.")
        return None

    features = get_predictors(df)
    if top_features is not None:
        features = [f for f in top_features if f in features]

    if len(features) < 2:
        print("Skipped:", title, "— fewer than two predictors.")
        return None

    work = df[features + [class_col]].dropna()
    if len(work) < 3:
        return None

    Xz = StandardScaler().fit_transform(work[features])
    pca = PCA(n_components=2)
    Z = pca.fit_transform(Xz)
    ev = pca.explained_variance_ratio_

    fig, ax = plt.subplots(figsize=(9,7))
    classes = work[class_col].astype(str)
    for cls in classes.unique():
        mask = classes == cls
        ax.scatter(Z[mask,0], Z[mask,1], alpha=0.75, label=cls)

    ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}% variance)")
    ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}% variance)")
    ax.set_title(title)
    ax.legend(title="WQI Class")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(PLOTS / filename, dpi=300, bbox_inches="tight")
    plt.show()

    return pca, ev, features

gw_pca = pca_plot(gw_data, "Groundwater — PCA WQI Class Feature Space",
                   "GW_PCA_classification.png")
sw_pca = pca_plot(sw_data, "Surface Water — PCA WQI Class Feature Space",
                   "SW_PCA_classification.png")


## 8. PCA explained-variance graph

This graph shows how much of the total standardized-feature variance is captured by each principal component. It is a useful companion to the 2-D PCA scatter plot.


In [ ]:
def pca_variance_plot(df, title, filename):
    if df is None:
        return
    class_col = detect_class_column(df)
    features = get_predictors(df)
    if len(features) < 2:
        return

    work = df[features].dropna()
    if len(work) < 3:
        return

    Xz = StandardScaler().fit_transform(work)
    pca = PCA().fit(Xz)
    ev = pca.explained_variance_ratio_

    fig, ax = plt.subplots(figsize=(10,5))
    n = min(len(ev), 10)
    x = np.arange(1, n+1)
    ax.bar(x, ev[:n]*100)
    ax.plot(x, np.cumsum(ev[:n])*100, marker="o", label="Cumulative variance")
    ax.set_xlabel("Principal Component")
    ax.set_ylabel("Explained variance (%)")
    ax.set_title(title)
    ax.set_xticks(x)
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(PLOTS / filename, dpi=300, bbox_inches="tight")
    plt.show()

pca_variance_plot(gw_data, "Groundwater — PCA Explained Variance", "GW_PCA_explained_variance.png")
pca_variance_plot(sw_data, "Surface Water — PCA Explained Variance", "SW_PCA_explained_variance.png")


## 9. Chi-square (χ²) feature association

We use χ² only as an **exploratory association test**. Because water-quality variables are continuous,
each feature is first divided into 4 quantile groups. `Year` and non-water-quality columns are excluded.

- χ² score: strength of departure from independence.
- p-value: evidence against independence; p < 0.05 is commonly treated as statistically significant.
- This does **not** prove causation or replace model feature importance.


In [ ]:
from scipy.stats import chi2_contingency

def chi_scores(df):
    if df is None:
        return pd.DataFrame()

    cls = detect_class_column(df)
    if cls is None:
        print("No WQI class column found.")
        return pd.DataFrame()

    # Only numeric water-quality variables; do not use Year or WQI itself.
    cols = []
    for c in df.select_dtypes(include=np.number).columns:
        name = str(c).lower()
        if name in {"year", "wqi", "wqi_encoded"}:
            continue
        if "iron" in name:
            continue
        cols.append(c)

    rows = []

    for c in cols:
        x = df[[c, cls]].dropna()

        if x[c].nunique() < 2 or x[cls].nunique() < 2:
            continue

        try:
            x["group"] = pd.qcut(
                x[c],
                q=min(4, x[c].nunique()),
                duplicates="drop"
            )
            table = pd.crosstab(x["group"], x[cls])

            if table.shape[0] < 2 or table.shape[1] < 2:
                continue

            chi, p, dof, _ = chi2_contingency(table)

            rows.append({
                "Feature": c,
                "Chi2 Score": chi,
                "p-value": p,
                "Significant (p<0.05)": "Yes" if p < 0.05 else "No"
            })
        except Exception:
            continue

    return (
        pd.DataFrame(rows)
        .sort_values("Chi2 Score", ascending=False)
        .reset_index(drop=True)
    )

gw_chi = chi_scores(gw_data)
sw_chi = chi_scores(sw_data)

print("GROUNDWATER χ² SCORES")
display(gw_chi)

print("\nSURFACE WATER χ² SCORES")
display(sw_chi)

if not gw_chi.empty:
    gw_chi.to_csv(PLOTS / "GW_chi_square_scores.csv", index=False)

if not sw_chi.empty:
    sw_chi.to_csv(PLOTS / "SW_chi_square_scores.csv", index=False)


In [ ]:
def combined_bar_plots(gw, sw, gw_chi, sw_chi, filename):
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))

    # 1. Groundwater model performance
    if gw is not None and not gw.empty:
        x = np.arange(len(gw))
        w = 0.18
        for j, metric in enumerate(["Accuracy", "Precision", "Recall", "F1", "MCC"]):
            if metric in gw:
                axes[0,0].bar(x + (j-2)*w, gw[metric], w, label=metric)
        axes[0,0].set_xticks(x)
        axes[0,0].set_xticklabels(gw["Model"], rotation=30, ha="right")
        axes[0,0].set_ylim(0, 1.05)
        axes[0,0].set_ylabel("Score")
        axes[0,0].set_title("Groundwater — Testing Performance")
        axes[0,0].legend(fontsize=8)
        axes[0,0].grid(axis="y", alpha=0.25)

    # 2. Surface water model performance
    if sw is not None and not sw.empty:
        x = np.arange(len(sw))
        w = 0.18
        for j, metric in enumerate(["Accuracy", "Precision", "Recall", "F1", "MCC"]):
            if metric in sw:
                axes[0,1].bar(x + (j-2)*w, sw[metric], w, label=metric)
        axes[0,1].set_xticks(x)
        axes[0,1].set_xticklabels(sw["Model"], rotation=30, ha="right")
        axes[0,1].set_ylim(0, 1.05)
        axes[0,1].set_ylabel("Score")
        axes[0,1].set_title("Surface Water — Testing Performance")
        axes[0,1].legend(fontsize=8)
        axes[0,1].grid(axis="y", alpha=0.25)

    # 3. Groundwater chi-square
    if gw_chi is not None and not gw_chi.empty:
        s = gw_chi.head(10).sort_values("Chi2 Score")
        axes[1,0].barh(s["Feature"], s["Chi2 Score"])
        axes[1,0].set_xlabel("χ² score")
        axes[1,0].set_title("Groundwater — Top χ² Associations")

    # 4. Surface water chi-square
    if sw_chi is not None and not sw_chi.empty:
        s = sw_chi.head(10).sort_values("Chi2 Score")
        axes[1,1].barh(s["Feature"], s["Chi2 Score"])
        axes[1,1].set_xlabel("χ² score")
        axes[1,1].set_title("Surface Water — Top χ² Associations")

    fig.suptitle("WQI Classification — Model Performance and χ² Feature Analysis",
                 fontsize=17, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(PLOTS / filename, dpi=300, bbox_inches="tight")
    plt.show()

combined_bar_plots(
    gw_test, sw_test, gw_chi, sw_chi,
    "ALL_bar_plots.png"
)


## 10. PCA after χ² feature ranking — exploratory companion

This optional view uses the **top χ²-ranked predictors** and then performs standardization + PCA. It is useful for visually checking whether the strongest univariate class associations also produce visible class separation.

Again, this is **exploratory only**. The reported classifiers were not refit using these selected features.


In [ ]:
def top_features_from_chi(scores, k=6):
    if scores is None or scores.empty:
        return []
    return scores.head(k)["Feature"].tolist()

gw_top = top_features_from_chi(gw_chi, 6)
sw_top = top_features_from_chi(sw_chi, 6)

print("GW top χ² features:", gw_top)
print("SW top χ² features:", sw_top)

pca_plot(
    gw_data,
    "Groundwater — PCA Using Top χ² Features",
    "GW_PCA_top_chi2_features.png",
    top_features=gw_top
)

pca_plot(
    sw_data,
    "Surface Water — PCA Using Top χ² Features",
    "SW_PCA_top_chi2_features.png",
    top_features=sw_top
)


## 11. Final figure checklist for the research paper

### Core classification figures
1. Original GW/SW WQI class distribution
2. Combined all-classifier confusion matrix PNG
3. Combined model-performance + χ² bar-plot PNG
4. Model-complexity radar (conceptual)
5. Original GW/SW WQI class distribution.
2. Testing Accuracy / Precision / Recall / F1 / MCC comparison.
3. Confusion matrix for the selected/best classifier (and optionally supplementary matrices for all classifiers).
4. Multiclass ROC–AUC for the selected/best classifier when probabilities are available.

### Diagnostics / exploratory figures
5. Training-vs-testing accuracy gap.
6. PCA class-space plot.
7. PCA explained-variance plot.
8. χ² feature-association score table + top-feature bar chart.
9. PCA using top χ² features (optional supplementary figure).

### Methodological caution
Do not present PCA or χ² as evidence that the classifier itself achieved a particular accuracy. Use the **held-out test metrics** for the main performance claim.
